# Risk-Controlled Explainable Defect Inspection

**Certified defect-escape guarantees with faithfulness-gated human triage, across five industrial datasets.**

---

## The problem

A deployed inspection model outputs a confidence score. Nobody can say what that
number means. "0.7" is not a probability of anything, it is not comparable
between two models, and it is not comparable between Tuesday and Thursday on the
same model. So the score threshold on a real production line is set by a
technician turning a dial until the reject pile looks about right.

That is the actual state of practice, and it is why a plant cannot answer the
only question that matters to its customer: **what fraction of defective parts
do we ship?**

## The claim

Given a held-out calibration set that is exchangeable with production, we
produce an operating point with a finite-sample, distribution-free guarantee

$$\mathbb{E}\big[\text{escape rate}\big] \;\le\; \alpha$$

that holds for **any** detector, **any** dataset size, and requires **no**
distributional assumption. When the target $\alpha$ is not achievable by the
detector, the system **refuses to issue a certificate** instead of quietly
returning a threshold that cannot deliver it.

## Contributions

| | contribution | where |
|---|---|---|
| **C1** | Instance-level escape risk control with per-class severity budgets (Mondrian) | §5, §6 |
| **C2** | Faithfulness-gated triage: explanation quality as a *certified gating variable*, minimising human review load subject to the escape guarantee | §8, §9 |
| **C3** | Drift-aware recalibration under production distribution shift | §7 |
| **C4** | Complementary Channel Encoding (CCE) — a zero-cost input encoding, reported with an honest ablation | §4 |

## How to run

Select the **`vqaenv`** kernel and run top to bottom. Every cell prints its
inputs and outputs; every processing stage is rendered before and after.
Figures are written to `figures/` as 300-dpi PDF and PNG.

## 0 · Environment, configuration, reproducibility

In [ ]:
import os, sys, json, glob, warnings, platform, random
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebook" else "."))
ROOT = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebook" else ".")
os.chdir(ROOT)

import numpy as np, cv2, torch, matplotlib.pyplot as plt
from src.paper import figures as F
F.set_style()

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"root        : {ROOT}")
print(f"python      : {platform.python_version()}")
print(f"torch       : {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu         : {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}")
print(f"opencv      : {cv2.__version__}")
print(f"seed        : {SEED}")

DATASETS = ["neu", "gc10", "pcb", "magnetic_tile", "kolektor"]
ALPHAS   = [0.01, 0.05, 0.10, 0.20]
os.makedirs("figures", exist_ok=True)

### System architecture

Each block is a section below. The calibration block is disjoint from training —
that disjointness is what makes the guarantee valid rather than decorative.

In [ ]:
fig = F.architecture_diagram(); F.save(fig, "fig01_architecture"); plt.show()

## 1 · Datasets

Five public industrial datasets spanning five different manufacturing domains.

Two of them (Magnetic-Tile, KolektorSDD2) are **load-bearing for a specific
reason**: they are the only ones containing genuine defect-free parts. NEU, GC10
and PCB have a defect in *every* image, so on those the auto-accept branch of a
triage policy can never be correct and the escape/review trade-off is not
measurable at all.

In [ ]:
import json
rows = []
for name in DATASETS:
    d = json.load(open(f"data/processed/{name}_coco.json"))
    neg = sum(1 for i in d["images"] if i.get("n_boxes", 1) == 0)
    ws = [a["bbox"][2] / next(im["width"] for im in d["images"] if im["id"] == a["image_id"])
          for a in d["annotations"][:400]]
    rows.append(dict(dataset=name, images=len(d["images"]), boxes=len(d["annotations"]),
                     classes=len(d["categories"]), defect_free=neg,
                     mean_box_w_pct=round(100 * float(np.mean(ws)), 1)))
import pandas as pd
df_data = pd.DataFrame(rows)
print(df_data.to_string(index=False))
print(f"\nTOTAL: {df_data.images.sum()} images, {df_data.boxes.sum()} boxes, "
      f"{df_data.classes.sum()} classes, {df_data.defect_free.sum()} defect-free")

### Sample images with ground truth, one row per dataset

In [ ]:
fig, axes = plt.subplots(len(DATASETS), 4, figsize=(11, 2.5 * len(DATASETS)))
for r, name in enumerate(DATASETS):
    d = json.load(open(f"data/processed/{name}_coco.json"))
    imgs = {i["id"]: i for i in d["images"]}
    by = {}
    for a in d["annotations"]:
        by.setdefault(a["image_id"], []).append(a["bbox"])
    ids = list(by)[:4]
    for c in range(4):
        ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if c < len(ids):
            im = cv2.cvtColor(cv2.imread(imgs[ids[c]]["file_name"]), cv2.COLOR_BGR2RGB)
            ax.imshow(im)
            for x, y, w, h in by[ids[c]]:
                ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec=F.OKABE[1], lw=1.5))
        if c == 0:
            ax.set_ylabel(name, fontsize=10, rotation=0, ha="right", va="center", labelpad=42)
fig.suptitle("Ground-truth annotations across the five domains", y=1.0)
fig.tight_layout(); F.save(fig, "fig02_dataset_samples"); plt.show()

### Class distribution and defect scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
counts = {n: json.load(open(f"data/processed/{n}_coco.json")) for n in DATASETS}
axes[0].bar(DATASETS, [len(counts[n]["images"]) for n in DATASETS], color=F.OKABE[0], label="images")
axes[0].bar(DATASETS, [sum(1 for i in counts[n]["images"] if i.get("n_boxes",1)==0) for n in DATASETS],
            color=F.OKABE[2], label="defect-free")
axes[0].set_ylabel("count"); axes[0].set_title("Dataset size and negative availability"); axes[0].legend()

for i, n in enumerate(DATASETS):
    d = counts[n]; imgs = {im["id"]: im for im in d["images"]}
    fr = [100*a["bbox"][2]*a["bbox"][3]/(imgs[a["image_id"]]["width"]*imgs[a["image_id"]]["height"])
          for a in d["annotations"]]
    axes[1].hist(fr, bins=40, range=(0, 40), histtype="step", label=n, color=F.OKABE[i])
axes[1].set_xlabel("defect area (% of frame)"); axes[1].set_ylabel("boxes")
axes[1].set_title("Defect scale — PCB is a small-object regime"); axes[1].legend(fontsize=8)
fig.tight_layout(); F.save(fig, "fig03_dataset_stats"); plt.show()

## 2 · Splits — why calibration is held out

Three disjoint blocks: **train** (60%), **cal** (15%), **test** (25%), stratified
by each image's rarest class so that rare defects and defect-free parts appear in
all three.

The calibration block is the load-bearing detail. Conformal risk control requires
calibration data that is exchangeable with test data **and untouched by model
fitting**. Reusing a validation set that early stopping or checkpoint selection
looked at breaks exchangeability and silently voids the guarantee — the numbers
would still print, they would simply be wrong.

In [ ]:
rows = []
for n in DATASETS:
    for s in ["train", "cal", "test"]:
        p = f"data/processed/splits/{n}_{s}.json"
        if os.path.exists(p):
            d = json.load(open(p))
            rows.append(dict(dataset=n, split=s, images=len(d["images"]),
                             boxes=len(d["annotations"]),
                             clean=sum(1 for i in d["images"] if i.get("n_boxes",1)==0)))
df_split = pd.DataFrame(rows)
print(df_split.pivot(index="dataset", columns="split", values="images").to_string())

## 3 · Photometric analysis — what the sensor actually gives us

Before any modelling, a fact that determines the input design.

In [ ]:
from src.data.photometric import to_gray
print("Are these images colour, or grey replicated into three channels?\n")
print(f"{'dataset':16s} {'shape':18s} {'ch0==ch1':>9s} {'ch1==ch2':>9s}  verdict")
print("-" * 74)
CHANNEL_REDUNDANT = {}
for name in DATASETS:
    d = json.load(open(f"data/processed/{name}_coco.json"))
    s01 = s12 = n = 0; shp = ""
    for info in d["images"][:60]:
        im = cv2.imread(info["file_name"], cv2.IMREAD_UNCHANGED)
        if im is None: continue
        shp = str(im.shape); n += 1
        if im.ndim == 2: s01 += 1; s12 += 1
        else:
            s01 += int(np.array_equal(im[...,0], im[...,1]))
            s12 += int(np.array_equal(im[...,1], im[...,2]))
    red = (s01 == n and s12 == n); CHANNEL_REDUNDANT[name] = red
    print(f"{name:16s} {shp:18s} {s01}/{n:<7d} {s12}/{n:<7d}  "
          f"{'GREY x3 — 2 channels are dead weight' if red else 'genuine colour'}")
print(f"\n{sum(CHANNEL_REDUNDANT.values())} of {len(DATASETS)} datasets waste two thirds of every input tensor.")

### Per-stage effect of each photometric operation

Every stage rendered before and after, with the intensity histogram that
explains *why* it moved.

In [ ]:
from src.data.photometric import (apply_clahe, high_frequency_residual,
                                  complementary_channels, local_contrast, entropy)
d = json.load(open("data/processed/neu_coco.json"))
imgs = {i["id"]: i for i in d["images"]}
by = {}
for a in d["annotations"]: by.setdefault(a["image_id"], []).append(a["bbox"])
iid = list(by)[7]
img = cv2.cvtColor(cv2.imread(imgs[iid]["file_name"]), cv2.COLOR_BGR2RGB)
boxes = [[x, y, x+w, y+h] for x, y, w, h in by[iid]]

enc, stages = complementary_channels(img, return_stages=True)
fig = F.image_strip(stages, title=f"CCE stages — {os.path.basename(imgs[iid]['file_name'])}", boxes=boxes)
F.save(fig, "fig04_cce_stages"); plt.show()

print(f"{'stage':26s} {'local contrast':>15s} {'entropy (bits)':>15s}")
for k, v in stages.items():
    print(f"{k:26s} {local_contrast(v):15.3f} {entropy(v):15.3f}")

In [ ]:
fig = F.histogram_panel({k: v for k, v in stages.items() if k != "04_cce_composite"},
                        "Intensity distribution at each CCE stage")
F.save(fig, "fig05_cce_histograms"); plt.show()
fig = F.cce_diagram(); F.save(fig, "fig06_cce_concept"); plt.show()

## 4 · Complementary Channel Encoding (C4)

CLAHE **as a substitution** is a bad trade — measured below. CLAHE **as a
complement** is free. The dead channels get spent on representations the grey
channel does not already contain:

| channel | content | preserves |
|---|---|---|
| ch0 | raw intensity | intensity-domain separability, which CLAHE destroys |
| ch1 | CLAHE | local contrast / edge energy |
| ch2 | $I - G_\sigma * I$ | surface texture and roughness |

Measured with the **multivariate Fisher discriminant**

$$J = (\mu_d - \mu_b)^\top S_w^{-1} (\mu_d - \mu_b)$$

which accounts for inter-channel covariance — so a channel that merely
duplicates another contributes **nothing** to $J$. That property is exactly what
makes it the right test here.

In [ ]:
from src.data.photometric import multivariate_separability

def _rep3(im):   g = to_gray(im); return np.stack([g, g, g], -1)
def _clahe3(im): g = apply_clahe(to_gray(im), 2.0, 8); return np.stack([g, g, g], -1)

def _sample(name, k=40):
    d = json.load(open(f"data/processed/{name}_coco.json"))
    im_map = {i["id"]: i for i in d["images"]}; b = {}
    for a in d["annotations"]: b.setdefault(a["image_id"], []).append(a["bbox"])
    out = []
    for iid in sorted(b)[:k]:
        im = cv2.imread(im_map[iid]["file_name"])
        if im is None: continue
        out.append((cv2.cvtColor(im, cv2.COLOR_BGR2RGB),
                    [[x, y, x+w, y+h] for x, y, w, h in b[iid]]))
    return out

ENC = {"grey x3 (standard)": _rep3, "CLAHE x3 (naive)": _clahe3,
       "CCE [raw|clahe|hf]": lambda im: complementary_channels(im)}
sep = {e: [] for e in ENC}
for n in DATASETS:
    s = _sample(n)
    for e, fn in ENC.items():
        sep[e].append(float(np.mean([multivariate_separability(fn(im), b) for im, b in s])))

print(f"{'encoding':22s} " + " ".join(f"{n[:11]:>12s}" for n in DATASETS))
print("-" * 88)
base = sep["grey x3 (standard)"]
for e, vals in sep.items():
    if e.startswith("grey"): print(f"{e:22s} " + " ".join(f"{v:12.4f}" for v in vals))
    else: print(f"{e:22s} " + " ".join(f"{100*(v-b)/max(b,1e-9):+11.1f}%" for v, b in zip(vals, base)))

fig = F.grouped_bars(DATASETS, {e: v for e, v in sep.items()},
                     "Fisher separability $J$", "Input encoding — multivariate separability")
fig.axes[0].set_yscale("log")
F.save(fig, "fig07_cce_separability"); plt.show()

### Ablation — does the separability gain survive into detection accuracy?

Separability is a **proxy**. Only a training ablation settles it, and the result
is reported here whatever it says.

In [ ]:
res = []
for f in ["runs/results_yolov8s.json", "runs/results_rtdetr.json"]:
    if os.path.exists(f): res += json.load(open(f))
key = lambda r: (r["model"], r["dataset"])
base_r = {key(r): r for r in res if r.get("encoding", "baseline") == "baseline"}
cce_r  = {key(r): r for r in res if r.get("encoding") == "cce"}

rows = []
for k in sorted(set(base_r) & set(cce_r)):
    b, c = base_r[k], cce_r[k]
    rows.append(dict(model=k[0].replace(".pt",""), dataset=k[1],
                     baseline=round(b["test_mAP50"], 4), cce=round(c["test_mAP50"], 4),
                     delta_pp=round(100*(c["test_mAP50"]-b["test_mAP50"]), 2)))
df_cce = pd.DataFrame(rows)
if len(df_cce):
    print(df_cce.to_string(index=False))
    print(f"\nCCE better on {int((df_cce.delta_pp>0).sum())}/{len(df_cce)} paired comparisons; "
          f"mean delta {df_cce.delta_pp.mean():+.2f} pp")
    fig = F.grouped_bars(list(df_cce.dataset), {"baseline": list(df_cce.baseline), "CCE": list(df_cce.cce)},
                         "test mAP@0.5", "CCE ablation (paired, same model/dataset/seed)")
    F.save(fig, "fig08_cce_ablation"); plt.show()
else:
    print("no completed CCE pairs yet")

## 5 · Detectors

**RT-DETR is the main method; YOLOv8s is a comparison baseline only.**

The choice is not about leaderboard position. RT-DETR is **NMS-free**. NMS is a
non-differentiable, score-dependent filter whose behaviour changes with box
density — and conformal calibration is a statement about precisely that score
distribution. An end-to-end set-prediction head emits one score per object with
no post-hoc suppression, so the calibration target is the model's own output
rather than an artefact of the filter applied after it.

In [ ]:
rows = [dict(model=r["model"].replace(".pt",""), dataset=r["dataset"],
             encoding=r.get("encoding","baseline"), imgsz=r["imgsz"], epochs=r["epochs"],
             mAP50=round(r["test_mAP50"],4), mAP50_95=round(r["test_mAP50_95"],4),
             minutes=round(r["train_seconds"]/60,1)) for r in res]
df_det = pd.DataFrame(rows).sort_values(["dataset","model","encoding"])
print(df_det.to_string(index=False))

In [ ]:
piv = df_det[df_det.encoding=="baseline"].pivot_table(index="dataset", columns="model", values="mAP50")
series = {m: [piv[m].get(d, np.nan) for d in DATASETS] for m in piv.columns}
fig = F.grouped_bars(DATASETS, series, "test mAP@0.5", "Detector comparison (baseline encoding)")
F.save(fig, "fig09_detector_comparison"); plt.show()

### Training dynamics

In [ ]:
curves = {}
for p in sorted(glob.glob("runs/detect/runs/detect/*/results.csv")):
    tag = os.path.basename(os.path.dirname(p))
    try:
        c = pd.read_csv(p); c.columns = [x.strip() for x in c.columns]
        col = [x for x in c.columns if "mAP50(B)" in x and "95" not in x]
        if col: curves[tag] = (c.index.values + 1, c[col[0]].values)
    except Exception: pass
if curves:
    sel = dict(list(curves.items())[:6])
    fig = F.curve_panel(sel, "epoch", "val mAP@0.5", "Training dynamics", figsize=(7.5, 4))
    F.save(fig, "fig10_training_curves"); plt.show()

## 6 · Conformal risk control — the core contribution

### The loss

For image $i$ at score threshold $\lambda$, the **defect escape loss** is the
fraction of true defects that no surviving prediction covers:

$$L_i(\lambda) \;=\; \frac{\big|\{\,g \in \mathrm{GT}_i \;:\; \nexists\, p,\ s_p \ge \lambda \ \wedge\ \mathrm{IoU}(g,p) \ge \tau \,\}\big|}{|\mathrm{GT}_i|}$$

Bounded in $[0,1]$, and **non-increasing as $\lambda$ decreases** — the
monotonicity that the CRC theorem requires. The implementation *checks* this
rather than assuming it, and raises if violated, because a silently non-monotone
loss returns a threshold with no guarantee attached.

### The calibration

Following Angelopoulos, Bates, Fisch, Lei & Schuster (ICLR 2024):

$$\hat{R}_n(\lambda) = \frac{1}{n}\sum_{i=1}^{n} L_i(\lambda), \qquad
\hat{\lambda} = \inf\Big\{\lambda : \tfrac{n}{n+1}\hat{R}_n(\lambda) + \tfrac{B}{n+1} \le \alpha\Big\}$$

giving $\mathbb{E}[L_{n+1}(\hat\lambda)] \le \alpha$. The $B/(n+1)$ term is the
finite-sample correction that makes the bound hold for the *next* part rather
than only on average over the calibration set.

### Two things this does **not** say

1. It bounds the **expectation** over calibration draws. A single split can land
   above $\alpha$ without violating the theorem. High-probability control is a
   different guarantee and needs Learn-then-Test — which is what §9 uses.
2. It assumes **exchangeability**. A production line that drifts breaks it, which
   is exactly what §7 measures.

In [ ]:
from src.risk.conformal import (conformal_risk_control, escape_threshold_grid,
                                build_calibration_losses, RiskNotAchievable)
risk = json.load(open("runs/risk_results.json"))
rows = []
for name, v in sorted(risk.items()):
    for a, e in sorted(v["repeated"].items(), key=lambda kv: float(kv[0])):
        af = float(a)
        rows.append(dict(dataset=name.replace("_yolov8s",""), alpha=af,
                         mean_escape=(None if e["n_issued"]==0 else round(e["mean_escape_test"],4)),
                         std=(None if e["n_issued"]==0 else round(e["std"],4)),
                         refusal_rate=round(e["refusal_rate"],3),
                         holds=(None if e["n_issued"]==0 else bool(e["mean_escape_test"] <= af + 1e-9))))
df_risk = pd.DataFrame(rows)
print(df_risk.to_string(index=False))
iss = df_risk[df_risk.holds.notna()]
print(f"\nGUARANTEE HELD ON {int(iss.holds.sum())}/{len(iss)} ISSUED OPERATING POINTS")
print(f"({int((df_risk.refusal_rate==1.0).sum())} operating points refused outright — alpha unreachable)")

### Validation figure

Points on or below the diagonal are the guarantee holding. Open markers are
**refusals** — the system declining to certify a target it cannot meet. Those
are results, not missing data; omitting them would overstate coverage.

In [ ]:
names = sorted(risk)
fig, axes = plt.subplots(1, len(names), figsize=(3.5*len(names), 3.6))
if len(names) == 1: axes = [axes]
for ax, name in zip(axes, names):
    sub = df_risk[df_risk.dataset == name.replace("_yolov8s","")]
    a = sub.alpha.values
    o = np.array([np.nan if v is None else v for v in sub.mean_escape], float)
    ref = np.array([r == 1.0 for r in sub.refusal_rate])
    lim = max(a.max(), np.nanmax(o) if np.any(~np.isnan(o)) else a.max())*1.15
    ax.plot([0,lim],[0,lim], ls="--", c="0.5", lw=1.1)
    ax.fill_between([0,lim],[0,lim],[lim,lim], color=F.OKABE[1], alpha=0.07)
    ok = ~np.isnan(o)
    ax.errorbar(a[ok], o[ok], yerr=np.array([0 if s is None else s for s in sub['std']],float)[ok],
                fmt="o", color=F.OKABE[0], capsize=3, ms=5)
    if ref.any(): ax.plot(a[ref], a[ref], "o", mfc="none", mec=F.OKABE[3], ms=8, mew=1.5)
    ax.set_xlim(0,lim); ax.set_ylim(0,lim); ax.set_aspect("equal")
    ax.set_title(name.replace("_yolov8s",""), fontsize=10)
    ax.set_xlabel(r"target $\alpha$")
axes[0].set_ylabel("observed escape rate")
fig.suptitle("Conformal risk control holds across all five domains (60 repeated splits)", y=1.04)
fig.tight_layout(); F.save(fig, "fig11_risk_coverage"); plt.show()

## 7 · Drift-aware recalibration (C3)

Exchangeability is the one assumption CRC cannot do without, and a production
line violates it routinely: a new steel coil, a lamp ageing, a camera nudged.
This section measures what the guarantee does under induced shift, and whether
recalibration restores it.

In [ ]:
if "drift" in str(list(risk.values())[0].keys()):
    print("drift results loaded from runs/risk_results.json")
for name, v in sorted(risk.items()):
    if "drift" in v:
        print(name, json.dumps(v["drift"])[:300])
print("\n(If empty, §7 is populated by scripts/run_all_experiments.py)")

## 8 · Faithfulness-audited explanations (C2, part 1)

Most XAI in inspection stops at "here is a heatmap, it looks right". That is
unfalsifiable. Here every explanation is scored:

| metric | reference | direction |
|---|---|---|
| deletion AUC | Petsiuk et al., RISE (BMVC 2018) | lower better |
| insertion AUC | same | higher better |
| pointing game | Zhang et al. (IJCV 2018) | higher better |
| energy pointing | Wang et al., Score-CAM (CVPRW 2020) | higher better |
| model-randomisation sanity | Adebayo et al. (NeurIPS 2018) | low correlation required |

The composite score is what §9 gates on — which is why it must be **per image**,
not a dataset average.

In [ ]:
from src.xai.faithfulness import (energy_pointing_game, pointing_game, box_area_fraction,
                                  faithfulness_score, insertion_curve, deletion_curve,
                                  normalized_auc)
print("Metric behaviour on controlled synthetic maps (validates the implementation):")
hm_perfect = np.zeros((100,100)); hm_perfect[40:60,40:60] = 1.0
hm_uniform = np.ones((100,100))
hm_off     = np.zeros((100,100)); hm_off[0:20,0:20] = 1.0
bx = [[40,40,60,60]]
for nm, h in [("mass on defect", hm_perfect), ("uniform (no info)", hm_uniform), ("mass off target", hm_off)]:
    print(f"  {nm:20s} energy={energy_pointing_game(h,bx):.3f}  "
          f"pointing={str(pointing_game(h,bx)):5s}  faithfulness={faithfulness_score(h,bx):.3f}")

### Audited faithfulness on real detector outputs

In [ ]:
import glob  # local import: the setup cell does not export it
# Audited faithfulness. compute_faithfulness.py writes one file per dataset,
# runs/faithfulness_<dataset>.json, each a list of per-image records.
fp_list = sorted(glob.glob("runs/faithfulness_*.json"))
if not fp_list:
    print("no faithfulness cache; run:  python scripts/compute_faithfulness.py 25 neu")
else:
    frames = []
    for fp in fp_list:
        ds = os.path.basename(fp).replace("faithfulness_", "").replace(".json", "")
        rows = json.load(open(fp))
        if not rows:
            continue
        d = pd.DataFrame(rows)
        d.insert(0, "dataset", ds)
        frames.append(d)

    df_f = pd.concat(frames, ignore_index=True)
    n_img = len(df_f)

    summary = (df_f.groupby("dataset")
               .agg(images=("faithfulness", "size"),
                    insertion_auc=("insertion_auc", "mean"),
                    energy_pointing=("energy_pointing", "mean"),
                    chance_level=("chance_level", "mean"),
                    pointing_game=("pointing_game", "mean"),
                    faithfulness=("faithfulness", "mean"))
               .reset_index())
    # Energy pointing is only meaningful against the chance level set by how much
    # of the frame the boxes cover; the raw number alone says nothing.
    summary["lift_over_chance"] = summary.energy_pointing / summary.chance_level
    print(summary.round(4).to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    axes[0].hist(df_f.faithfulness, bins=20, color=F.OKABE[0], edgecolor="white")
    axes[0].set_title("per-image faithfulness"); axes[0].set_xlabel("score")

    axes[1].scatter(df_f.chance_level, df_f.energy_pointing, s=26,
                    color=F.OKABE[0], alpha=.75, edgecolor="white", linewidth=.6)
    lim = [0, max(df_f.chance_level.max(), df_f.energy_pointing.max()) * 1.08]
    axes[1].plot(lim, lim, ls="--", lw=1.2, color="#888888")
    axes[1].set_xlim(lim); axes[1].set_ylim(lim)
    axes[1].set_xlabel("chance level (box area fraction)")
    axes[1].set_ylabel("energy pointing")
    axes[1].set_title("above the diagonal = better than chance")

    axes[2].scatter(df_f.insertion_auc, df_f.faithfulness, s=26,
                    color=F.OKABE[1], alpha=.75, edgecolor="white", linewidth=.6)
    axes[2].set_xlabel("insertion AUC"); axes[2].set_ylabel("composite faithfulness")
    axes[2].set_title("causal vs localisation agreement")
    for a in axes:
        a.grid(alpha=.25)
    fig.suptitle(f"Explanation faithfulness audit  ({n_img} images, "
                 f"{len(summary)} dataset(s))", y=1.04)
    fig.tight_layout()
    F.save(fig, "fig14_faithfulness"); plt.show()

    above = (df_f.energy_pointing > df_f.chance_level).mean()
    print(f"\n{above*100:.0f}% of maps concentrate more attribution inside the "
          f"defect boxes than box area alone would give.")
    print("Faithfulness is what gates the triage policy in section 9, which is why")
    print("it is measured per image rather than reported as a dataset average.")


## 9 · Risk-controlled triage (C2, part 2)

Three-way decision per part:

$$
\text{decide}(s,\phi)=
\begin{cases}
\textsf{AUTO\_ACCEPT} & s < \lambda_{\text{lo}}\\
\textsf{AUTO\_REJECT} & s \ge \lambda_{\text{hi}} \ \wedge\ \phi \ge \phi_{\min}\\
\textsf{HUMAN\_REVIEW} & \text{otherwise}
\end{cases}
$$

Chosen by constrained optimisation:

$$\min\ \mathbb{E}[\text{review load}] \quad \text{s.t.} \quad \mathbb{E}[\text{escape}] \le \alpha \ \text{ certified w.p. } 1-\delta$$

The certification uses **Learn-then-Test** with Hoeffding–Bentkus $p$-values and
family-wise error control. That choice is load-bearing: because LTT certifies
*all* configurations simultaneously, we may then select the review-load minimiser
among them **without a second multiplicity penalty**. Selecting on the same data
used for a per-configuration test would be exactly the winner's-curse error that
voids the guarantee.

In [ ]:
from src.risk.triage import build_grid, fit_triage_policy, evaluate_policy, NoCertifiableConfig
tp = "runs/triage_results.json"
if os.path.exists(tp):
    tr = json.load(open(tp))
    rows = []
    for name, v in tr.items():
        for e in (v if isinstance(v, list) else [v]):
            rows.append(dict(dataset=name, alpha=e.get("alpha"),
                             escape=e.get("escape_rate_per_image"), review=e.get("review_load"),
                             auto=e.get("auto_decision_rate")))
    df_tri = pd.DataFrame(rows); print(df_tri.to_string(index=False))
    if len(df_tri):
        fig = F.pareto_plot(list(df_tri.review), list(df_tri.escape),
                            alpha=float(df_tri.alpha.iloc[0]), labels=list(df_tri.dataset))
        F.save(fig, "fig12_triage_pareto"); plt.show()
else:
    print("run scripts/run_all_experiments.py to populate this section")

## 10 · Closed-loop inspection agent

The agent runs the loop a plant actually executes: route each part under the
certified policy, watch the score stream for drift, recalibrate or escalate when
drift fires, and spend a limited labelling budget on the parts that most tighten
the certificate.

**Where the language model sits, and why it sits there.** Every decision below —
accept, review, reject, recalibrate, escalate — is made by the conformal
controller, never by the LLM. A language model cannot carry a statistical
certificate; put one in the decision path and you get fluent prose attached to an
unbounded error rate, which is the exact failure this project argues against. So
Qwen is given the one job it is genuinely good at: turning certified machine
state into a shift report an operator can act on.

The separation is testable. If the model is offline, wrong, or hallucinating, the
inspection outcome is byte-identical — only the prose changes. The cell prints
which backend answered so that claim is visible rather than asserted.


In [ ]:
# ---- live closed-loop episode on real cached predictions --------------------
from src.agentic.inspector import InspectionAgent, run_episode
from src.agentic import llm_narrator as NARR

AGENT_DS = "kolektor"      # 89% clean parts -> the only regime where auto-accept is meaningful
AGENT_ALPHA = 0.10

cal_p = f"runs/preds/{AGENT_DS}_yolov8s_cal.json"
test_p = f"runs/preds/{AGENT_DS}_yolov8s_test.json"

if not (os.path.exists(cal_p) and os.path.exists(test_p)):
    print(f"prediction cache missing for {AGENT_DS}; run scripts/run_all_experiments.py")
else:
    cal_rec = json.load(open(cal_p))["records"]
    test_rec = json.load(open(test_p))["records"]

    agent = InspectionAgent(cal_rec, alpha=AGENT_ALPHA, drift_window=150)
    print(f"calibrated on {len(cal_rec)} held-out parts -> lambda={agent.state.lam:.4f} "
          f"certified={agent.state.certified}")

    run_episode(agent, test_rec)
    summ = agent.summary() if callable(getattr(agent, "summary", None)) else agent.state.as_dict()
    st = summ if isinstance(summ, dict) else agent.state.as_dict()

    print(f"\nepisode over {st.get('n_parts', len(test_rec))} parts")
    for k in ("n_accept", "n_review", "n_reject", "n_escapes", "n_defective_seen",
              "drift_alarms", "recalibrations", "labels_requested"):
        if k in st:
            print(f"  {k:20s} {st[k]}")
    for k in ("review_load", "auto_rate", "escape_rate"):
        if k in st:
            print(f"  {k:20s} {st[k]:.4f}")
    print(f"  escape budget alpha  {AGENT_ALPHA}  -> "
          f"{'WITHIN BUDGET' if st.get('escape_rate', 1) <= AGENT_ALPHA else 'OVER BUDGET'}")

    # ---- decision trace: first parts through the loop -----------------------
    if getattr(agent, "log", None):
        print("\nfirst 8 decisions:")
        for e in agent.log[:8]:
            print("  ", json.dumps(e)[:150])

    # ---- LLM narration over certified state ---------------------------------
    print("\n" + "=" * 70)
    ns = NARR.status()
    print(f"LLM backend : {ns['selected_model'] or 'none'}  (mode={ns['mode']})")
    print(f"ollama up   : {ns['ollama_running']}")
    print("=" * 70)

    stats = {"dataset": AGENT_DS, "alpha": AGENT_ALPHA,
             "n_parts": st.get("n_parts"), "auto_accept": st.get("n_accept"),
             "human_review": st.get("n_review"), "auto_reject": st.get("n_reject"),
             "escapes": st.get("n_escapes"), "drift_alarms": st.get("drift_alarms"),
             "recalibrations": st.get("recalibrations"),
             "review_load": round(st.get("review_load", 0), 4),
             "escape_rate": round(st.get("escape_rate", 0), 4)}

    rep = NARR.shift_report(stats)
    print(f"\n--- SHIFT REPORT  (llm={rep['llm']}, source={rep['source']}) ---")
    print(rep["text"])

    # A routed part, explained to the operator
    routed = next((r for r in test_rec
                   if agent._max_score(r) >= agent.state.lam), None)
    if routed is not None:
        rec_view = {"image_id": routed.get("image_id"),
                    "max_score": round(agent._max_score(routed), 4),
                    "n_detections": len(routed.get("pred_scores", []))}
        pol = {"lambda_lo": round(agent.state.lam, 4), "phi": agent.phi}
        ex = NARR.explain_routing(rec_view, pol)
        print(f"\n--- WHY THIS PART WAS ROUTED  (llm={ex['llm']}) ---")
        print(ex["text"])

    print("\nNote: the numbers above come from the certified controller. The LLM only")
    print("describes them - disabling it changes the prose, not a single decision.")


## 11 · Positioning against published work

| work | year | task | guarantee | triage | XAI audit |
|---|---|---|---|---|---|
| DSAT | 2025 | steel defect detection | none | no | no |
| SH-DETR | 2025 | steel defect detection | none | no | no |
| Conformal Segmentation for Surface Defects | 2025 | **segmentation** | FDR / FNR, pixel-level | no | no |
| Conformal Object Detection (SeqCRC) | 2025 | detection, general | risk control | no | no |
| **this work** | — | **detection, 5 domains** | **instance escape + per-class budgets** | **yes, certified** | **yes, quantitative** |

Detection accuracy is **not** the claim here — reference numbers on NEU-DET are
83.1 (DSAT) and 83.0 (SH-DETR) mAP@0.5. What is new is a certificate on the
quantity a plant is actually accountable for, a triage policy that minimises
human cost subject to it, and explanation faithfulness promoted from decoration
to a gating variable.

In [ ]:
lit = pd.DataFrame([
    dict(method="DSAT (2025)", mAP50=0.8314, source="Sci. Reports"),
    dict(method="SH-DETR (2025)", mAP50=0.8303, source="PLOS ONE"),
    dict(method="HCT-Det (2025)", mAP50=0.7950, source="Sensors"),
    dict(method="YOLOv8n+Swin (2026)", mAP50=0.7760, source="Sci. Reports"),
])
ours = df_det[(df_det.dataset=="neu")].sort_values("mAP50", ascending=False)
if len(ours):
    lit = pd.concat([lit, pd.DataFrame([dict(method=f"this work ({ours.iloc[0].model})",
                                             mAP50=float(ours.iloc[0].mAP50), source="—")])])
print(lit.to_string(index=False))
fig = F.grouped_bars(list(lit.method), {"mAP@0.5": list(lit.mAP50)}, "mAP@0.5",
                     "NEU-DET — detection accuracy in context", figsize=(8, 3.4))
fig.axes[0].tick_params(axis="x", rotation=18)
F.save(fig, "fig13_literature"); plt.show()

## 12 · Reproducibility and honest limitations

### Pipeline

```
src/data/prepare_datasets.py     raw -> per-dataset COCO
src/data/splits_and_yolo.py      stratified train/cal/test -> YOLO layout
src/data/build_cce_dataset.py    CCE-encoded copies
src/evaluation/train_baselines.py  detectors
scripts/run_risk_experiment.py   conformal calibration + validation
```

### Limitations, stated plainly

1. **Exchangeability is an assumption, not a fact.** Under real drift the
   guarantee degrades; §7 measures this rather than assuming it away.
2. **CRC bounds an expectation.** Per-split excursions above $\alpha$ are
   expected. High-probability control requires LTT (§9).
3. **Detection accuracy is mid-field**, not state of the art. The contribution is
   the certificate, not the detector.
4. **CCE is reported honestly** — separability gains are large, the mAP effect is
   near neutral. A pretrained backbone already learns CLAHE-like filters, so
   hand-crafting them into the input adds little.
5. **Single GPU, single seed** for most cells. Multi-seed variance is reported
   where available and is a stated gap elsewhere.

In [ ]:
print("Artifacts produced by this notebook:")
for p in sorted(glob.glob("figures/*.pdf")): print("  ", p)
print(f"\n{len(glob.glob('figures/*.pdf'))} vector figures at 300 dpi")